In [56]:
def data_describer(dataframe):
    # Get the description of the dataframe
    description = dataframe.describe()
    
    # Convert the description to a string with column names
    description_str = "Data Description:\n"
    for col in description.columns:
        description_str += f"\nColumn: {col}\n"
        description_str += description[col].to_string() + "\n"
    
    # Write the description to a file
    with open("df_description.txt", "w", encoding="utf-8") as f:
        f.write(description_str)
    return description_str

In [57]:
from langchain.agents import AgentExecutor, Tool, create_react_agent
from langchain import hub
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain, SequentialChain
from langchain_community.llms import Ollama
#from OprFuncs import data_infer, extract_code, extract_questions
import pandas as pd

# Initialize Ollama
llm = Ollama(model="llama3.2:3b")

In [58]:
dataframe = pd.read_csv("Test_Datasets/WorldCupMatches.csv")

In [59]:
from OprFuncs import data_infer
dataframe = dataframe
data_info = data_infer(dataframe)
data_summary = data_describer(dataframe)
data_head = dataframe.head().to_string

In [60]:
guidelines = """▼ Chart Selection Matrix
| Scenario                          | Chart Type      |
|------------------------------------|-----------------|
| Time series analysis               | Line chart      |
| Comparing >3 categories            | Bar chart       |
| Distribution of continuous data    | Histogram       |
| Part-to-whole relationships        | Pie chart       |
| Correlation between 2 variables    | Scatter plot    |
| Multivariate comparison            | Heatmap         |
| Geographical data                  | Choropleth      |

▲ Special Cases:
- Use box plots for statistical distributions
- Use stacked bars for cumulative totals
- Avoid pie charts when >5 categories"""

In [61]:
chart_selection_prompt = PromptTemplate(
    input_variables=["data_info", "data_sample", "data_summary", "guidelines", "question"],
    template="""
    You are a data analyst. You are provided with:
        1. Dataset metadata: {data_info}
        2. Dataset sample: {data_sample}
        3. Dataset summary: {data_summary}
        Analyze this question to determine the best chart type:
    Question: {question}
    Respond ONLY with the chart type name (line, bar, pie, etc.), your chart type selection is based on knowledge from {guidelines}"""
)
chart_chain = LLMChain(llm=llm, prompt=chart_selection_prompt)

def chart_selector(input_text):
    return chart_chain.run(
        question=input_text,
        data_info=data_info,
        data_sample=data_head,
        data_summary=data_summary,
        guidelines=guidelines  # Corrected parameter name
    )

In [62]:
code_gen_prompt = PromptTemplate(
input_variables=["question", "chart_type", "data_info", "data_sample", "data_summary"],
    template="""
    You are provided with:
        1. Dataset metadata: {data_info}
        2. Dataset sample: {data_sample}
        3. Dataset summary: {data_summary}
    Generate matplotlib code for {chart_type} chart answering:
    Question: {question}
    Include sample data, labels, and plt.show()"""
)
code_chain = LLMChain(llm=llm, prompt=code_gen_prompt)

def code_generator(inputs):
    return code_chain.run(
        question=inputs["question"],
        chart_type=inputs["chart_type"],
        data_info=data_info,
        data_sample=data_head,
        data_summary=data_summary  
    )

In [63]:
tools = [
    Tool(
        name="ChartSelector",
        func=chart_selector,
        description="Determine appropriate chart type for a question"
    ),
Tool(
    name="CodeGenerator",
    func=lambda x: code_generator({
        "question": x.split("|")[0] if "|" in x else x,
        "chart_type": x.split("|")[1] if "|" in x else "bar"  # Default to bar chart
    }),
    description="Generate matplotlib code for specified chart type"
)
]

In [64]:
agent_prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, agent_prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

c:\Users\hasso\AppData\Local\Programs\Python\Python312\Lib\site-packages\langsmith\client.py:354: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [ ]:
question = "Show the most teams played as home team off all time"
result = agent_executor.invoke({
    "input": f"""Follow these steps:
    1. Analyze question: {question}
    2. Select chart type
    3. Generate appropriate code
    
    Use format: "question|chart_type" for code generation"""
})

print(result["output"])



> Entering new AgentExecutor chain...
Action: ChartSelector
Action Input: "most teams played as home team off all time"Based on the question "most teams played as home team off all time", I would recommend a Bar chart.

The reason is that we want to compare the number of times each team has played as the home team, which can be represented by bar heights. This comparison is suitable for a bar chart, which allows us to visualize categorical data (team names) on the x-axis and numerical data (number of times played as home team) on the y-axis.

Additionally, this scenario does not require time series analysis or correlation between variables, making Line chart or Scatter plot less suitable. A Pie chart is also not ideal since there are more than 5 categories (teams), which could lead to a cluttered and difficult-to-read chart.Action: CodeGenerator
Action Input: "most teams played as home team off all time|bar"Here's the Python code that uses matplotlib to generate a bar chart showing t